# UR3 Dataset Inspector

Load and explore a generated HDF5 trajectory dataset.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))
import h5py
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path('../data')
# Change this to the dataset you want to inspect
H5_PATH = DATA_DIR / 'single_trajectory.h5'
TRAJ_NAME = 'trajectory_000'


In [ ]:
# Load data
with h5py.File(H5_PATH, 'r') as f:
    print('Keys:', list(f.keys()))
    grp = f[TRAJ_NAME]
    print('\nAttributes:', dict(grp.attrs))
    data = {k: grp[k][:] for k in grp.keys()}

print('\nFields and shapes:')
for k, v in data.items():
    print(f'  {k}: {v.shape} dtype={v.dtype}')


In [ ]:
# Plot joint positions, velocities, accelerations
t = data['time']
joint_names = ['shoulder_pan', 'shoulder_lift', 'elbow', 'wrist_1', 'wrist_2', 'wrist_3']

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

for j in range(6):
    axes[0].plot(t, data['q'][:, j], label=joint_names[j])
axes[0].set_ylabel('q (rad)')
axes[0].set_title('Joint Positions')
axes[0].legend(fontsize=7)

for j in range(6):
    axes[1].plot(t, data['q_dot'][:, j])
axes[1].set_ylabel('qdot (rad/s)')
axes[1].set_title('Joint Velocities')

for j in range(6):
    axes[2].plot(t, data['q_dot_dot'][:, j])
axes[2].set_ylabel('qdotdot (rad/s²)')
axes[2].set_title('Joint Accelerations')
axes[2].set_xlabel('Time (s)')

plt.tight_layout()
plt.show()


In [ ]:
# Plot end-effector position and velocity
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

for d, lbl in enumerate(['x', 'y', 'z']):
    axes[0].plot(t, data['ee_pos'][:, d], label=lbl)
axes[0].set_ylabel('Position (m)')
axes[0].set_title('End-Effector Position')
axes[0].legend()

for d, lbl in enumerate(['vx', 'vy', 'vz']):
    axes[1].plot(t, data['ee_lin_vel'][:, d], label=lbl)
axes[1].set_ylabel('Velocity (m/s)')
axes[1].set_title('End-Effector Linear Velocity (Jacobian)')
axes[1].legend()
axes[1].set_xlabel('Time (s)')

plt.tight_layout()
plt.show()


In [ ]:
# Check rotation matrix validity
R_flat = data['ee_rotmat']
max_det_err = 0.0
max_orth_err = 0.0
for i in range(len(R_flat)):
    R = R_flat[i].reshape(3, 3)
    max_det_err = max(max_det_err, abs(np.linalg.det(R) - 1.0))
    max_orth_err = max(max_orth_err, np.max(np.abs(R.T @ R - np.eye(3))))
print(f'Max det error: {max_det_err:.2e}')
print(f'Max orthogonality error: {max_orth_err:.2e}')
print('Rotation matrix: VALID' if max_det_err < 1e-4 else 'INVALID')
